# Setup — load model, tokenizer, and a pushT batch

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
from dreamerv4uwm.datasets import ShardedHDF5Dataset
from dreamerv4uwm.models.utils import load_tokenizer
from dreamerv4uwm.models.utils import load_denoiser
# DATA_PATH = "/home/mim-server/datasets/soar_data_sharded"
# DATA_PATH = "/home/mim-server/datasets/Finger/H5/combined"
DATA_PATH = "/scratch/rk4342/datasets/pushT/demo"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resolution = (256, 256)

In [ ]:
from hydra import initialize, compose
from omegaconf import OmegaConf
with initialize(version_base=None, config_path="../scripts/config"):
    cfg = compose(config_name="dynamics/g1.yaml")

In [ ]:
cfg.denoiser.train_reward_model = True

In [ ]:
is_2x_temporal = True

if is_2x_temporal:
    cfg.denoiser.layer_types = ["spatial", "temporal", "spatial", "temporal"]
    dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/wm-policy-2x-temporal-293044.pt"
else:
    cfg.denoiser.layer_types = ["spatial", "spatial", "spatial", "temporal"]
    dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/wm-policy-1x-temporal-293044.pt"


> **NB:** for shortcut sampling, set `dynamics_ckpt` in the cell below to a
> shortcut-trained checkpoint (`checkpoints/dynamics/pushT/with-shortcuts/...`)
> before running the loader.


In [ ]:
# dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/all-2x-temporal-485204.pt"
dynamics_ckpt = "/scratch/rk4342/projects/dreamerV4-UWM/checkpoints/dynamics/pushT-reward-demo-play-mix/51060.pt"

In [ ]:
tokenizer_ckpt="/scratch/rk4342/projects/dreamer-v4/checkpoints/tokenizer_ckpts/soar.pt"
cfg.dynamics_ckpt = dynamics_ckpt
cfg.tokenizer_ckpt=tokenizer_ckpt
denoiser = load_denoiser(cfg, device, max_num_forward_steps=300)
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300)
tokenizer = tokenizer.eval().cuda()
denoiser = denoiser.eval().cuda()

In [ ]:
import mediapy
from torch.nn.functional import interpolate

dataset = ShardedHDF5Dataset(
        data_dir=DATA_PATH,
        window_size=64,
        stride=1,
        split='train',
        train_fraction=0.9,
        split_seed=123,
    )

# Arbitrary index into that episode
batch = dataset[torch.randint(len(dataset), (1,)).item()]
# imgs = batch["image"][:,[2, 1, 0], :, :]  # (T, C, H, W)
imgs = batch["image"]  # (T, C, H, W)
actions = batch["action"][:,:cfg.denoiser.n_actions]  # (1, T, N_lat, D_lat)
# actions=torch.zeros_like(actions)
imgs = interpolate(imgs, resolution).to(device=device)[None] # resize to tokenizer resolution

with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs)
        imgs_recon = tokenizer.decode(latents)

from mediapy import show_video
def plotVideo(video):
    imgs = video.cpu().permute(0,2,3,1).to(torch.float32).numpy()*255
    imgs = imgs.astype('uint8')
    mediapy.show_video(imgs, fps=10)

plotVideo(imgs_recon[0])
plotVideo(imgs[0])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plotSnapshots(video, n_frames=5, border_width=2, figsize=None):
    """Plot n_frames evenly-spaced snapshots from a video tensor, separated by black borders.

    Args:
        video: (T, C, H, W) tensor with values in [0, 1].
        n_frames: number of frames to sample.
        border_width: width of black separator lines in pixels.
    """
    T = video.shape[0]
    indices = np.linspace(0, T - 1, n_frames, dtype=int)
    frames = video[indices].cpu().permute(0, 2, 3, 1).to(torch.float32).numpy()
    frames = np.clip(frames * 255, 0, 255).astype(np.uint8)
    H, W, C = frames.shape[1], frames.shape[2], frames.shape[3]
    border = np.zeros((H, border_width, C), dtype=np.uint8)
    parts = []
    for i, f in enumerate(frames):
        if i > 0:
            parts.append(border)
        parts.append(f)
    strip = np.concatenate(parts, axis=1)
    if figsize is None:
        figsize = (n_frames * 3, 3)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(strip)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def plotActions(actions, figsize=None):
    act = actions.cpu().float().numpy() if hasattr(actions, 'cpu') else np.array(actions)
    figsize = (3, 3)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.plot(act[:, 0], linewidth=3, label=r'$\mathbf{v_x}$')
    ax.plot(act[:, 1], linewidth=3, label=r'$\mathbf{v_y}$')
    ax.set_ylim(-1., 1.)
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    ax.tick_params(axis='y', labelsize=16)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.legend(fontsize=16)
    plt.tight_layout()
    plt.show()


# Unified Shortcut Sampler — All 5 UWM Modes

The samplers below target the **shortcut** UWM denoiser (trained with `compute_bootstrap_uwm_loss`). Unlike the flow-matching samplers, which always use the finest step (`step_idx = 0 ↔ d_min`), the shortcut samplers set `step_idx = get_step_index(1/num_diffusion_steps, num_noise_levels)` so the denoiser takes a big dyadic step per Euler update.

Make sure `dynamics_ckpt` points to a shortcut-trained checkpoint (see cell above:
`checkpoints/dynamics/pushT/with-shortcuts/...`).


In [ ]:
from dreamerv4uwm.sampling import unified_shortcut_sampler

num_pred_steps = 16
num_diffusion_steps = 8   # power of two; shortcut big-step
latents_ctx = latents[:, :4, ...].clone()
T_ctx = latents_ctx.shape[1]


## 1. World Model (`wm`) — shortcut
Given context obs and **all** actions, predict future observations using the shortcut big-step sampler.

In [ ]:
all_actions_wm = actions[:T_ctx + num_pred_steps][None].clone()
all_actions_wm[:, T_ctx:, 0] = 0.0
all_actions_wm[:, T_ctx:, 1] = 0.0

with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    pred_obs_wm_sc, _ = unified_shortcut_sampler(
        denoiser, latents_ctx, mode='wm',
        all_actions=all_actions_wm,
        num_pred_steps=num_pred_steps,
        num_diffusion_steps=num_diffusion_steps,
    )
    with torch.no_grad():
        img_pred_wm_sc = tokenizer.decode(pred_obs_wm_sc)

plotVideo(img_pred_wm_sc[0].to(torch.float32))
plotSnapshots(img_pred_wm_sc[0].to(torch.float32), n_frames=4)
plotActions(all_actions_wm[0, T_ctx:T_ctx+num_pred_steps])


## 2. Policy (`policy`) — shortcut
Given context obs + context actions, jointly predict future observations and future actions.

In [ ]:
ctx_actions_policy = actions[:T_ctx][None].cuda()

with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    pred_obs_pol_sc, pred_act_pol_sc = unified_shortcut_sampler(
        denoiser, latents_ctx, mode='policy',
        ctx_actions=ctx_actions_policy,
        num_pred_steps=num_pred_steps,
        num_diffusion_steps=num_diffusion_steps,
    )
    with torch.no_grad():
        img_pred_pol_sc = tokenizer.decode(pred_obs_pol_sc)

plotVideo(img_pred_pol_sc[0].to(torch.float32))
plotSnapshots(img_pred_pol_sc[0].to(torch.float32))
plotActions(pred_act_pol_sc[0].cpu().float().numpy())


## 3. Video Prediction (`video`) — shortcut
Given context obs only (actions are pure noise), predict future observations.

In [ ]:
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    pred_obs_vid_sc, _ = unified_shortcut_sampler(
        denoiser, latents_ctx, mode='video',
        ctx_actions=ctx_actions_policy,  # only used to infer n_act
        num_pred_steps=num_pred_steps,
        num_diffusion_steps=num_diffusion_steps,
    )
    with torch.no_grad():
        img_pred_vid_sc = tokenizer.decode(pred_obs_vid_sc)

plotVideo(img_pred_vid_sc[0].to(torch.float32))
plotSnapshots(img_pred_vid_sc[0].to(torch.float32))


## 4. Inverse Dynamics (`id`) — shortcut
Given **all** observations (clean), predict actions across all frames.

In [ ]:
all_latents_id = latents[:, :T_ctx + num_pred_steps, ...]

with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    _, pred_act_id_sc = unified_shortcut_sampler(
        denoiser, latents_ctx, mode='id',
        ctx_actions=ctx_actions_policy,
        all_latents=all_latents_id,
        num_pred_steps=num_pred_steps,
        num_diffusion_steps=num_diffusion_steps,
    )

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
gt_act = actions[T_ctx:T_ctx + num_pred_steps].cpu().float().numpy()
pr_act = pred_act_id_sc[0].cpu().float().numpy()
axes[0].set_title("Ground-truth future actions")
axes[0].plot(gt_act)
axes[1].set_title("Predicted future actions (inverse dynamics, shortcut)")
axes[1].plot(pr_act)
for ax in axes:
    ax.set_xlabel("time step")
plt.tight_layout()
plt.show()


## 5. Diffusion Forcing (`forcing`) — shortcut
Given context obs + context actions (both slightly noised), jointly denoise future obs and actions.

In [ ]:
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    pred_obs_forc_sc, pred_act_forc_sc = unified_shortcut_sampler(
        denoiser, latents_ctx, mode='forcing',
        ctx_actions=ctx_actions_policy,
        num_pred_steps=num_pred_steps,
        num_diffusion_steps=num_diffusion_steps,
    )
    with torch.no_grad():
        img_pred_forc_sc = tokenizer.decode(pred_obs_forc_sc)

plotVideo(img_pred_forc_sc[0].to(torch.float32))
plotSnapshots(img_pred_forc_sc[0].to(torch.float32))

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
gt_act = actions[T_ctx:T_ctx + num_pred_steps].cpu().float().numpy()
pr_act = pred_act_forc_sc[0].cpu().float().numpy()
axes[0].set_title("Ground-truth future actions")
axes[0].plot(gt_act)
axes[1].set_title("Predicted future actions (forcing, shortcut)")
axes[1].plot(pr_act)
for ax in axes:
    ax.set_xlabel("time step")
plt.tight_layout()
plt.show()
